# 01 Path EPC, Linewidth, And Relaxation Time

This notebook computes fixed-k plus q-path EPC using DeePTB Python APIs. It is the DeePTB-native counterpart of the path/scattering part of the dftbephy Graphene workflow.

In [ ]:
from pathlib import Path
import json
import numpy as np

from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons

ROOT = Path.cwd()
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

# Local reference assets available on this workstation.
DFTBEPHY_GRAPHENE = Path("/Users/aisiqg/Desktop/work/github/dftbephy/examples/Graphene")
MATSCI_SK = Path("/Users/aisiqg/Desktop/work/github/matsci-0-3")

STRUCTURE = ROOT / "graphene.vasp"
if not STRUCTURE.exists():
    STRUCTURE = DFTBEPHY_GRAPHENE / "opt" / "geo_end.gen"
PHONOPY_YAML = DFTBEPHY_GRAPHENE / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DFTBEPHY_GRAPHENE / "phonons" / "FORCE_SETS"

# matsci-0-3 C-C.skf contains the carbon SK data used by this example model.
BASIS = {"C": ["2s", "2p"]}

for path in [MATSCI_SK, STRUCTURE, PHONOPY_YAML, FORCE_SETS]:
    if not path.exists():
        raise FileNotFoundError(path)

model = DFTBSK(
    basis=BASIS,
    skdata=str(MATSCI_SK),
    overlap=True,
    dtype="float64",
    device="cpu",
    interp_method="smooth_intp",
    r_max=6.0,
)
system = TBSystem(data=str(STRUCTURE), calculator=model)
print(system.atoms)
print(system.model.name, system.model.basis)


In [ ]:
def regularize_tiny_negative_frequencies(phonons, tol=1e-3):
    """Clip tiny negative phonopy frequencies caused by acoustic numerical noise."""
    frequencies = np.array(phonons.frequencies, copy=True)
    min_frequency = float(np.min(frequencies))
    if min_frequency < -tol:
        raise ValueError(
            f"Found phonon frequency {min_frequency} THz below tolerance {-tol} THz; "
            "this looks like a real imaginary mode, not acoustic numerical noise."
        )
    clipped = int(np.count_nonzero(frequencies < 0.0))
    frequencies[frequencies < 0.0] = 0.0
    if clipped:
        phonons = Phonons(
            qpoints=phonons.qpoints,
            frequencies=frequencies,
            eigenvectors=phonons.eigenvectors,
            masses=phonons.masses,
            cell=phonons.cell,
            scaled_positions=phonons.scaled_positions,
            metadata={
                **phonons.metadata,
                "negative_frequency_regularization": "clipped_to_zero",
                "negative_frequency_tolerance_thz": tol,
                "negative_frequency_min_before_clip_thz": min_frequency,
                "negative_frequency_clipped_count": clipped,
            },
        )
    return phonons


In [ ]:
def floor_path_linewidth_for_relaxation(linewidth_data, floor=1e-12):
    """Apply a small positive linewidth floor before tau=hbar/(2*linewidth).

    Zero linewidth means infinite relaxation time. This helper is for example
    plotting/diagnostics; production analysis should inspect zero-scattering
    channels and convergence before choosing a floor.
    """
    from dptb.postprocess.unified.eph import LinewidthPathData

    linewidth = np.maximum(np.asarray(linewidth_data.linewidth, dtype=float), floor)
    absorption = np.maximum(np.asarray(linewidth_data.absorption, dtype=float), 0.0)
    emission = np.maximum(np.asarray(linewidth_data.emission, dtype=float), 0.0)
    return LinewidthPathData(
        linewidth=linewidth,
        absorption=absorption,
        emission=emission,
        path_axis=linewidth_data.path_axis,
        path_coordinates=linewidth_data.path_coordinates,
        path_segments=linewidth_data.path_segments,
        band_indices=linewidth_data.band_indices,
        metadata={
            **linewidth_data.metadata,
            "linewidth_floor_for_relaxation": floor,
            "linewidth_floor_for_relaxation_unit": linewidth_data.metadata.get("linewidth_unit", "eV"),
            "linewidth_floor_for_relaxation_reason": "avoid_infinite_tau_in_example_notebook",
        },
    )


In [ ]:
def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


In [ ]:
from dptb.postprocess.unified.eph import (
    Phonons,
    compute_linewidth_path,
    compute_relaxation_time_path,
    compute_coupling_strength_summary,
)

phonons = regularize_tiny_negative_frequencies(Phonons.load_npz(WORK / "phonons_qpath.npz"))
kpoints = np.array([[0.0, 0.0, 0.0]], dtype=float)

# Choose bands near the Fermi level for your model/material. These are example indices.
bands = [0, 1, 2, 3]
displacement = 1e-3
chemical_potential = 0.0  # eV
temperature = 0.025       # kBT in eV
sigma = 0.01              # eV


## Compute q-path EPC

In [ ]:
epc_path = system.eph.compute_path(
    kpoints=kpoints,
    phonons=phonons,
    bands=bands,
    displacement=displacement,
    output_npz=WORK / "epc_path_data.npz",
    path_labels={"G": 0, "K": 2, "M": 3, "G2": 4},
)
print(epc_path.coupling_strength.shape)
print(epc_path.path_coordinates)
print(epc_path.metadata.keys())


## Path-resolved linewidth and relaxation time

In [ ]:
path_linewidth = compute_linewidth_path(
    epc_path,
    chemical_potential=chemical_potential,
    temperature=temperature,
    sigma=sigma,
    broadening="gaussian",
    mode_resolved=True,
)
path_linewidth.save_npz(WORK / "path_linewidth.npz")

path_linewidth_for_tau = floor_path_linewidth_for_relaxation(path_linewidth, floor=1e-12)
path_tau = compute_relaxation_time_path(path_linewidth_for_tau, sum_modes=True)
path_tau.save_npz(WORK / "path_relaxation_time.npz")

print("linewidth", path_linewidth.linewidth.shape, "min", float(np.min(path_linewidth.linewidth)))
print("tau", path_tau.relaxation_time.shape, "max", float(np.max(path_tau.relaxation_time)))


## Coupling summary

In [ ]:
summary = compute_coupling_strength_summary(epc_path)
(WORK / "path_coupling_summary.json").write_text(json.dumps(summary, indent=2, default=json_default))
print(json.dumps(summary["metadata"], indent=2, default=json_default))
